In [1]:
from langchain_openai import ChatOpenAI
from langchain_google_vertexai import ChatVertexAI

from wsd.load_data import load_data
from wsd.models import BinaryWSD, ClusterByMeaningModel, DummyComparator, chat_template
from linpub.metrics import accuracy

In [2]:
X, y = load_data(lang='en')

k = 200
X_test, y_test = X[:k], y[:k]

In [7]:
from wsd.models import Candidate
# 見る,VERB,bn:00088428v,-6060086627281818467,見,"あなたがたは,労働をして,あおぎ分けているのを見て,いつものように,労働をしているのか.すなわち,一日になって,午パスの講ピデにおいて,まきになった日にすぎないことがあろうか."
# 見る,VERB,bn:00093430v,-6060086627281818467,見る,彼はまた寄るべなきirを見るように求めた.
# 見る,VERB,bn:00092443v,-6060086627281818467,見,彼女たちはわれわれを見ようと計りました.
candidates = [
    Candidate(lemma='見る', pos='VERB', text='見',
              context="あなたがたは,労働をして,あおぎ分けているのを見て,いつものように,労働をしているのか.すなわち,一日になって,午パスの講ピデにおいて,まきになった日にすぎないことがあろうか."),
    Candidate(lemma='見る', pos='VERB', text='見る',
              context="彼はまた寄るべなきirを見るように求めた."),
    Candidate(lemma='見る', pos='VERB', text='見', context="彼女たちはわれわれを見ようと計りました"),
    Candidate(lemma='見る', pos='VERB', text='見', context="やって見ないか"),
    Candidate(lemma='見る', pos='VERB', text='見', context="この猫はおじさんに見える"),
]
ref = Candidate(lemma='tour', pos='NOUN', text='tour', context="La tour Eiffel")
can = Candidate(lemma='tour', pos='NOUN', text='tour',context="Le tour de France")
candidates = [ref, can]
comparator = ChatOpenAI(temperature=0, model="gpt-4o")\
    .with_structured_output(BinaryWSD)
comparator = ChatVertexAI(temperature=0, model="gemini-1.5-pro")\
    .with_structured_output(BinaryWSD)
model = ClusterByMeaningModel(comparator=comparator)
print(model.predict(candidates))

messages = chat_template.format_messages(
    word1=ref.text,
    lemma1=ref.lemma,
    context1=ref.context,
    word2=can.text,
    lemma2=can.lemma,
    context2=can.context,
)
for m in messages:
    print(m.content)
comparator.invoke(messages)

[-6729315772097194396, 2425996889655292726]
You are a linguist working on word sense disambiguation.
Compare the two words provided and return the probability that they have the same meaning.
word: tour; lemma: tour; context: La tour Eiffel
word: tour; lemma: tour; context: Le tour de France


BinaryWSD(probability=0.0)

In [ ]:
gpt = ChatOpenAI(temperature=0, model="gpt-4o")\
    .with_structured_output(BinaryWSD)
model1 = ClusterByMeaningModel(comparator=gpt)

gemini = ChatVertexAI(temperature=0, model="gemini-1.5-pro")\
    .with_structured_output(BinaryWSD)
model2 = ClusterByMeaningModel(comparator=gemini)

dummy = DummyComparator(probability=1)
model3 = ClusterByMeaningModel(comparator=dummy)

dummy = DummyComparator(probability=0)
model4 = ClusterByMeaningModel(comparator=dummy)

y_pred1 = model1.predict(X_test, verbose=True)
y_pred2 = model2.predict(X_test, verbose=True)
y_pred3 = model3.predict(X_test, verbose=True)
y_pred4 = model4.predict(X_test, verbose=True)

print(f"gpt-4o-mini:  {accuracy(y_pred1, y_test)}")
print(f"gemini1.5-flash: {accuracy(y_pred2, y_test)}")
print(f"dummy (always 1): {accuracy(y_pred3, y_test)}")
print(f"dummy (always 0): {accuracy(y_pred4, y_test)}")

  0%|          | 0/142 [00:00<?, ?it/s]Retrying langchain_google_vertexai.chat_models._completion_with_retry.<locals>._completion_with_retry_inner in 4.0 seconds as it raised NotFound: 404 Publisher Model `projects/linpub/locations/us-central1/publishers/google/models/gemini-1.5` not found..
Retrying langchain_google_vertexai.chat_models._completion_with_retry.<locals>._completion_with_retry_inner in 4.0 seconds as it raised NotFound: 404 Publisher Model `projects/linpub/locations/us-central1/publishers/google/models/gemini-1.5` not found..
Retrying langchain_google_vertexai.chat_models._completion_with_retry.<locals>._completion_with_retry_inner in 4.0 seconds as it raised NotFound: 404 Publisher Model `projects/linpub/locations/us-central1/publishers/google/models/gemini-1.5` not found..
Retrying langchain_google_vertexai.chat_models._completion_with_retry.<locals>._completion_with_retry_inner in 8.0 seconds as it raised NotFound: 404 Publisher Model `projects/linpub/locations/us-cen

KeyboardInterrupt: 

In [5]:
import pandas as pd

records = []
for yt, yp, x in zip(y_test, y_pred1, X_test):
    record = {'lemma': x.lemma, 'pos': x.pos, 'y': yt, 'y_pred': yp, 'text': x.text, 'context': x.context}
    records.append(record)
pd.DataFrame(records).sort_values(['lemma', 'pos']).to_csv('plop.csv', index=False)